In [ ]:
# =============================================================
# TAHAP 2: CASE REPRESENTATION
# Mata Kuliah Penalaran Komputer - CBR Korupsi Kerugian Keuangan Negara
# =============================================================

# Tahap 2: Case Representation
**Tujuan:** Representasikan setiap putusan dalam struktur data terorganisir
dengan mengekstrak metadata dan fitur teks penting.

In [ ]:
Import Library
import os
import re
import json
import glob
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [ ]:
Konfigurasi Path
BASE_DIR  = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) \
            if "__file__" in dir() else os.getcwd()
RAW_DIR   = os.path.join(BASE_DIR, "data", "raw")
PROC_DIR  = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(PROC_DIR, exist_ok=True)
print(f"✅ Path siap | RAW: {RAW_DIR} | PROCESSED: {PROC_DIR}")

In [ ]:
Fungsi Ekstraksi Metadata
def extract_no_perkara(text: str) -> str:
    """Ekstrak nomor perkara dari teks putusan."""
    patterns = [
        r"nomor\s*:?\s*([\d]+\s*/\s*pid\.sus\s*/\s*\d{4}(?:/[a-z]+\.?\w*)*)",
        r"nomor\s*:?\s*([\d]+\s*/\s*pid\s*/\s*\d{4}(?:/[a-z]+\.?\w*)*)",
        r"putusan\s+nomor\s*([\w/\.\-]+)",
        r"no\.\s*([\d]+\s*/\s*[\w\.]+/\d{4}(?:/[\w\.]+)*)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).strip().upper()
    return "TIDAK_DITEMUKAN"


def extract_tanggal(text: str) -> str:
    """Ekstrak tanggal putusan."""
    bulan_map = {
        "januari":"01","februari":"02","maret":"03","april":"04",
        "mei":"05","juni":"06","juli":"07","agustus":"08",
        "september":"09","oktober":"10","november":"11","desember":"12"
    }
    pat = r"(\d{1,2})\s+(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\s+(\d{4})"
    m = re.search(pat, text, re.IGNORECASE)
    if m:
        d, bln, y = m.group(1), m.group(2).lower(), m.group(3)
        return f"{y}-{bulan_map[bln]}-{int(d):02d}"
    return "TIDAK_DITEMUKAN"


def extract_jenis_perkara(text: str) -> str:
    """Jenis perkara selalu korupsi kerugian keuangan negara untuk dataset ini."""
    keywords = ["korupsi", "tipikor", "kerugian keuangan negara", "tindak pidana korupsi"]
    for kw in keywords:
        if kw in text.lower():
            return "Pidana Khusus - Korupsi Kerugian Keuangan Negara"
    return "Pidana Khusus - Korupsi Kerugian Keuangan Negara"


def extract_pasal(text: str) -> str:
    """Ekstrak pasal yang digunakan dalam putusan."""
    patterns = [
        r"(pasal\s+\d+(?:\s+ayat\s+\(\d+\))?\s+(?:undang-undang|uu)\s+(?:nomor\s+)?\d+\s+tahun\s+\d+)",
        r"(pasal\s+\d+\s+(?:huruf\s+[a-z]\s+)?(?:jo\.?\s*pasal\s+\d+\s+)*undang-undang.*?tipikor)",
        r"(pasal\s+\d+(?:\s+ayat\s+\(\d+\))?(?:\s+jo\.?\s*pasal\s+\d+(?:\s+ayat\s+\(\d+\))?)*)",
    ]
    found = set()
    for pat in patterns:
        matches = re.findall(pat, text, re.IGNORECASE)
        for m in matches[:3]:  # ambil maks 3
            found.add(m.strip())
    return " | ".join(list(found)[:3]) if found else "TIDAK_DITEMUKAN"


def extract_terdakwa(text: str) -> str:
    """Ekstrak nama terdakwa."""
    patterns = [
        r"terdakwa\s*:?\s*([A-Z][A-Z\s\.,]+?)(?:\n|;|,\s*(?:bin|binti|alias|als))",
        r"nama\s*lengkap\s*:?\s*([A-Z][A-Z\s\.,]+?)(?:\n|;)",
        r"(?:terdakwa|terpidana)\s+([A-Z][A-Z\s]+?)(?:\s+(?:bin|binti|alias|terbukti|telah))",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            nama = m.group(1).strip()
            if len(nama) > 3 and len(nama) < 100:
                return nama
    return "TIDAK_DITEMUKAN"


def extract_penuntut_umum(text: str) -> str:
    """Ekstrak nama penuntut umum / jaksa."""
    patterns = [
        r"penuntut\s+umum\s*:?\s*([A-Z][A-Z\s\.,]+?)(?:\n|;)",
        r"jaksa\s+penuntut\s+umum\s*:?\s*([A-Z][A-Z\s\.,]+?)(?:\n|;)",
        r"nama\s*:?\s*([A-Z][A-Z\s\.,]+?),?\s*s\.?h",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            nama = m.group(1).strip()
            if len(nama) > 3 and len(nama) < 100:
                return nama
    return "TIDAK_DITEMUKAN"


def extract_amar_putusan(text: str) -> str:
    """Ekstrak amar putusan (bagian 'mengadili/memutuskan')."""
    patterns = [
        r"(mengadili\s*:.*?)(?=\n\n|\Z)",
        r"(m\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i.*?)(?=\n\n|\Z)",
        r"(amar\s+putusan\s*:.*?)(?=\n\n|\Z)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.DOTALL)
        if m:
            amar = m.group(1).strip()
            # Batasi panjang
            return amar[:1000] if len(amar) > 1000 else amar
    # Fallback: cari kalimat hukum terdakwa
    m2 = re.search(
        r"((?:menghukum|menjatuhkan|memidana|menyatakan terdakwa terbukti).*?)(?=\n\n|\Z)",
        text, re.IGNORECASE | re.DOTALL
    )
    if m2:
        return m2.group(1).strip()[:800]
    return "TIDAK_DITEMUKAN"


def extract_ringkasan_fakta(text: str) -> str:
    """Ekstrak ringkasan fakta persidangan (dakwaan / kronologi)."""
    patterns = [
        r"(?:bahwa\s+terdakwa.*?)(?=menimbang|mengadili|\Z)",
        r"(?:fakta\s+(?:hukum|persidangan)\s*:.*?)(?=menimbang|mengadili|\Z)",
        r"(?:dakwaan\s*:.*?)(?=tuntutan|mengadili|\Z)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.DOTALL)
        if m:
            fakta = m.group(0).strip()
            return fakta[:1000] if len(fakta) > 1000 else fakta
    # Fallback: 3 paragraf pertama
    paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip()) > 50]
    return " ".join(paragraphs[:3])[:800]


def extract_pidana_penjara(text: str) -> str:
    """Ekstrak lama pidana penjara dari amar putusan."""
    patterns = [
        r"pidana\s+penjara\s+selama\s+([\d]+\s+(?:tahun|bulan|hari)(?:\s+[\d]+\s+(?:bulan|hari))?)",
        r"penjara\s+(?:selama\s+)?([\d]+\s+(?:tahun|bulan))",
        r"(\d+\s+(?:tahun|bulan))\s+(?:penjara|kurungan)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return "TIDAK_DITEMUKAN"


def extract_uang_pengganti(text: str) -> str:
    """Ekstrak jumlah uang pengganti kerugian negara."""
    patterns = [
        r"uang\s+pengganti\s+(?:sebesar\s+)?(?:rp\.?\s*)?([\d\.,]+(?:\s+(?:miliar|juta|ribu))?)",
        r"membayar\s+uang\s+pengganti\s+(?:rp\.?\s*)?([\d\.,]+(?:\s+(?:miliar|juta|ribu))?)",
        r"(?:rp\.?\s*|rupiah\s+)([\d\.,]+(?:\s+(?:miliar|juta|ribu))?)\s+(?:sebagai\s+)?uang\s+pengganti",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return "Rp " + m.group(1).strip()
    return "TIDAK_DITEMUKAN"


def count_words(text: str) -> int:
    return len(text.split())

In [ ]:
Fungsi: Label kategori putusan untuk klasifikasi
def label_putusan(amar: str) -> str:
    """
    Menghasilkan label sederhana dari amar putusan untuk keperluan klasifikasi.
    Label: 'bersalah' / 'bebas' / 'tidak_dapat_diterima'
    """
    amar_lower = amar.lower()
    if any(k in amar_lower for k in ["menyatakan terdakwa terbukti", "menghukum", "memidana", "pidana penjara"]):
        return "bersalah"
    elif any(k in amar_lower for k in ["membebaskan", "bebas dari dakwaan", "tidak terbukti"]):
        return "bebas"
    elif any(k in amar_lower for k in ["tidak dapat diterima", "tidak dapat diperiksa"]):
        return "tidak_dapat_diterima"
    return "bersalah"  # default untuk korupsi

In [ ]:
Pipeline Representasi: Proses semua file .txt
def run_representation_pipeline():
    print("\n" + "="*60)
    print("  TAHAP 2: CASE REPRESENTATION")
    print("="*60)

    txt_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.txt")))
    print(f"\n📁 Ditemukan {len(txt_files)} file teks di {RAW_DIR}")

    if not txt_files:
        print("❌ Tidak ada file .txt! Jalankan dulu Tahap 1.")
        return None

    records = []
    for idx, fpath in enumerate(txt_files, start=1):
        case_id = os.path.splitext(os.path.basename(fpath))[0]
        print(f"  [{idx}/{len(txt_files)}] Memproses {case_id}...", end=" ")

        with open(fpath, "r", encoding="utf-8") as f:
            text = f.read()

        no_perkara      = extract_no_perkara(text)
        tanggal         = extract_tanggal(text)
        jenis_perkara   = extract_jenis_perkara(text)
        pasal           = extract_pasal(text)
        terdakwa        = extract_terdakwa(text)
        penuntut        = extract_penuntut_umum(text)
        amar            = extract_amar_putusan(text)
        fakta           = extract_ringkasan_fakta(text)
        pidana          = extract_pidana_penjara(text)
        uang_pengganti  = extract_uang_pengganti(text)
        jumlah_kata     = count_words(text)
        label           = label_putusan(amar)

        records.append({
            "case_id"           : case_id,
            "no_perkara"        : no_perkara,
            "tanggal"           : tanggal,
            "jenis_perkara"     : jenis_perkara,
            "pasal"             : pasal,
            "terdakwa"          : terdakwa,
            "penuntut_umum"     : penuntut,
            "ringkasan_fakta"   : fakta,
            "amar_putusan"      : amar,
            "pidana_penjara"    : pidana,
            "uang_pengganti"    : uang_pengganti,
            "jumlah_kata"       : jumlah_kata,
            "label"             : label,
            "text_full"         : text,
        })
        print(f"✅ ({jumlah_kata} kata | label: {label})")

    df = pd.DataFrame(records)

    # Simpan CSV
    csv_path = os.path.join(PROC_DIR, "cases.csv")
    df.drop(columns=["text_full"]).to_csv(csv_path, index=False, encoding="utf-8-sig")

    # Simpan JSON (tanpa text_full agar ringkas)
    json_path = os.path.join(PROC_DIR, "cases.json")
    df.drop(columns=["text_full"]).to_json(json_path, orient="records", force_ascii=False, indent=2)

    # Simpan versi lengkap dengan text_full
    full_path = os.path.join(PROC_DIR, "cases_full.csv")
    df.to_csv(full_path, index=False, encoding="utf-8-sig")

    print(f"\n✅ Dataset tersimpan:")
    print(f"   CSV        : {csv_path}")
    print(f"   JSON       : {json_path}")
    print(f"   Full CSV   : {full_path}")
    print(f"\n📊 Statistik Dataset:")
    print(f"   Total kasus     : {len(df)}")
    print(f"   Distribusi label:\n{df['label'].value_counts().to_string()}")
    print(f"   Rata-rata kata  : {df['jumlah_kata'].mean():.0f}")
    print(f"   Kolom metadata  : {list(df.columns)}")
    print("="*60)

    return df

In [ ]:
Tampilkan Pratinjau
if __name__ == "__main__":
    df = run_representation_pipeline()
    if df is not None:
        print("\n📋 Pratinjau 3 baris pertama:")
        cols_preview = ["case_id","no_perkara","tanggal","terdakwa","label","jumlah_kata"]
        print(df[cols_preview].head(3).to_string(index=False))